# DeepLOB "Super Model" (DeepLOB Encoder + Seq2Seq Decoder)

This notebook implements the architecture inspired by the MHF paper.

**Architecture:**
1.  **Encoder:** DeepLOB (Convolutional Blocks + Inception Module + LSTM) to extract rich spatial features from the LOB.
2.  **Decoder:** Seq2Seq LSTM with Additive Attention to generate the 60-second midprice path step-by-step.


In [35]:
import numpy as np
from pathlib import Path
import numpy as np
import torch
import random
from utils.utils import *
from data.simulate_walk_the_book import *
from utils.datastuff import TrainCfg
from utils.train import train_val


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Fix randomness for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


device: cuda


In [36]:
# Paths and volume_to_fill
root = Path("data") if Path("data").exists() else Path.cwd()
import sys
if str(root / "src") not in sys.path:
    sys.path.append(str(root / "src"))

symbol_dir = root / "BTCUSDT"
X_path = symbol_dir / "X_train.parquet"
Y_path = symbol_dir / "y_train.parquet"
vol_path = symbol_dir / "vol_to_fill.txt"

volume_to_fill = None
if vol_path.exists():
    import re
    with open(vol_path) as f:
        m = re.search(r"([\d.]+)", f.read())
    if m:
        volume_to_fill = float(m.group(1))
print("volume_to_fill:", volume_to_fill)


volume_to_fill: 4.0


In [37]:
# DeepLOB only take LOB features as input
LOB_COLS = []
for i in range(1, 6):
    LOB_COLS.append(f"ask_price_{i}")
    LOB_COLS.append(f"ask_vol_{i}")
    LOB_COLS.append(f"bid_price_{i}")
    LOB_COLS.append(f"bid_vol_{i}")

FEATURE_COLS = LOB_COLS

# Verification print
print(f"CNN Input Width: 4 (Columns: Price/Vol/Price/Vol)")
print(f"CNN Input Height: 5 (Rows: Levels 1 through 5)")
print(f"Total Features: {len(FEATURE_COLS)}")

CNN Input Width: 4 (Columns: Price/Vol/Price/Vol)
CNN Input Height: 5 (Rows: Levels 1 through 5)
Total Features: 20


In [38]:
# --- Execution ---
config = TrainCfg(

    # Hyperparameters
    epochs = 50,
    batch_size = 32,
    lr = 1e-3,
    weight_decay = 1e-5,
    smooth_lambda = 0.01,
    
    # Windows
    input_window = 600,   # Look-back
    target_window = 60,   # Horizon
    val_ratio = 0.2,

    # Environment
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    
    # Features & Data
    x_path = X_path,
    y_path = Y_path,
    feature_cols = FEATURE_COLS,
)

model, scalers, val_loader, va_id_map = train_val(config)

Epoch 01 | Train: 0.242059 | Val: 0.471560
Epoch 02 | Train: 0.050343 | Val: 0.117507
Epoch 03 | Train: 0.033538 | Val: 0.016610
Epoch 04 | Train: 0.032665 | Val: 0.022815
Epoch 05 | Train: 0.042019 | Val: 0.031429
Epoch 06 | Train: 0.025245 | Val: 0.072765
Epoch 07 | Train: 0.035830 | Val: 0.021154
Epoch 08 | Train: 0.027248 | Val: 0.026205
Epoch 09 | Train: 0.030034 | Val: 0.034777
Epoch 10 | Train: 0.022360 | Val: 0.025003
Epoch 11 | Train: 0.019890 | Val: 0.004938
Epoch 12 | Train: 0.018040 | Val: 0.016590
Epoch 13 | Train: 0.018517 | Val: 0.019046
Epoch 14 | Train: 0.015037 | Val: 0.004727
Epoch 15 | Train: 0.011787 | Val: 0.019891
Epoch 16 | Train: 0.014426 | Val: 0.011760
Epoch 17 | Train: 0.013239 | Val: 0.017102
Epoch 18 | Train: 0.008277 | Val: 0.019079
Epoch 19 | Train: 0.007530 | Val: 0.002800
Epoch 20 | Train: 0.005893 | Val: 0.008167
Epoch 21 | Train: 0.004213 | Val: 0.003208
Epoch 22 | Train: 0.006949 | Val: 0.022797
Epoch 23 | Train: 0.005505 | Val: 0.007556
Epoch 24 | 

In [40]:
# Implementation error for the model

import pandas as pd
import numpy as np
from data.simulate_walk_the_book import simulate_walk_the_book

# --- 1. Generate predictions using val_loader ---
model.eval()
all_preds = []

with torch.no_grad():
    for xb, yb, target in val_loader:
        xb = xb.to(config.device)
        pred = model(xb, y_teacher=None)
        all_preds.append(pred.cpu().numpy())

val_preds = np.concatenate(all_preds, axis=0)  # Shape: [num_val_ids, 60]

# Get validation IDs
val_ids = [int(uid) for idx, uid in va_id_map.cpu().numpy()]
print(f"Validation predictions shape: {val_preds.shape}")
print(f"Number of validation instruments: {len(val_ids)}")

# --- 2. Reload raw Y validation data to simulate walking the book ---
Y_raw = pd.read_parquet(Y_path).sort_values(["anonymized_id", "time_in_hour"])
Y_val_raw = Y_raw[Y_raw["anonymized_id"].isin(val_ids)].copy()

# --- 3. Column definitions ---
ASK_PRICE_COLS = ['ask_price_1', 'ask_price_2', 'ask_price_3', 'ask_price_4', 'ask_price_5']
ASK_VOL_COLS = ['ask_vol_1', 'ask_vol_2', 'ask_vol_3', 'ask_vol_4', 'ask_vol_5']
BID_PRICE_COLS = ['bid_price_1', 'bid_price_2', 'bid_price_3', 'bid_price_4', 'bid_price_5']
BID_VOL_COLS = ['bid_vol_1', 'bid_vol_2', 'bid_vol_3', 'bid_vol_4', 'bid_vol_5']

# --- 4. Calculate MODEL implementation error ---
model_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Model-based positions
    mid_preds = val_preds[i]
    pred_close = mid_preds[-1]
    
    price_advantage = pred_close - mid_preds
    price_advantage = np.maximum(price_advantage, 0)
    
    if price_advantage.sum() > 0:
        weights = price_advantage / price_advantage.sum()
    else:
        weights = np.zeros(60)
        weights[-14:] = 1.0 / 14  # Fallback to TWAP
    
    model_positions = weights * volume_to_fill
    
    # Simulate
    model_vol, model_avg_price = simulate_walk_the_book(
        model_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if model_vol > 0 and not np.isnan(model_avg_price):
        impl_error = np.abs(model_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / model_vol)
        model_bps.append(impl_error * vol_penalty)

model_bps = np.array(model_bps)

print(f"\n{'='*50}")
print(f"MODEL IMPLEMENTATION ERROR")
print(f"{'='*50}")
print(f"Instruments evaluated: {len(model_bps)}")
print(f"Mean:   {model_bps.mean():.4f} bps")
print(f"Median: {np.median(model_bps):.4f} bps")
print(f"Std:    {model_bps.std():.4f} bps")
print(f"Min:    {model_bps.min():.4f} bps")
print(f"Max:    {model_bps.max():.4f} bps")
print(f"{'='*50}")

Validation predictions shape: (136, 60)
Number of validation instruments: 136

MODEL IMPLEMENTATION ERROR
Instruments evaluated: 136
Mean:   1.6622 bps
Median: 1.3850 bps
Std:    1.3608 bps
Min:    0.0000 bps
Max:    9.0855 bps


/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 55 level 5. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 48 level 3. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 48 level 4. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 48 level 5. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 21 level 3. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 21 level 4. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encounte

In [41]:
# Implementation error for the baseline (TWAP)

K_SECONDS = 14  # TWAP window: last K seconds

baseline_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Baseline TWAP positions
    baseline_positions = np.zeros(60)
    baseline_positions[-K_SECONDS:] = volume_to_fill / K_SECONDS
    
    # Simulate
    baseline_vol, baseline_avg_price = simulate_walk_the_book(
        baseline_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if baseline_vol > 0 and not np.isnan(baseline_avg_price):
        impl_error = np.abs(baseline_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / baseline_vol)
        baseline_bps.append(impl_error * vol_penalty)

baseline_bps = np.array(baseline_bps)

print(f"\n{'='*50}")
print(f"BASELINE (TWAP-{K_SECONDS}s) IMPLEMENTATION ERROR")
print(f"{'='*50}")
print(f"Instruments evaluated: {len(baseline_bps)}")
print(f"Mean:   {baseline_bps.mean():.4f} bps")
print(f"Median: {np.median(baseline_bps):.4f} bps")
print(f"Std:    {baseline_bps.std():.4f} bps")
print(f"Min:    {baseline_bps.min():.4f} bps")
print(f"Max:    {baseline_bps.max():.4f} bps")
print(f"{'='*50}")


BASELINE (TWAP-14s) IMPLEMENTATION ERROR
Instruments evaluated: 136
Mean:   1.3340 bps
Median: 1.1894 bps
Std:    1.1121 bps
Min:    0.0000 bps
Max:    9.0855 bps


/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 55 level 5. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 48 level 3. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 48 level 4. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 48 level 5. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 57 level 3. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 57 level 4. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encounte

In [ ]:
# Implementation error for the predictive scheduler

from predictive_scheduler import build_schedule_from_forecasts, ScheduleConfig

# Configure the scheduler
sched_cfg = ScheduleConfig(
    window=14,       # Only trade in last 14 seconds
    alpha=0.1,       # 10% model, 90% TWAP blend
    price_cap=3.0,   # Cap extreme scores
)

scheduler_bps = []

for i, anon_id in enumerate(val_ids):
    df_inst = Y_val_raw[Y_val_raw["anonymized_id"] == anon_id].sort_values("time_in_hour")
    
    if len(df_inst) != 60:
        continue
    
    ask_prices = df_inst[ASK_PRICE_COLS].to_numpy()
    ask_vols = df_inst[ASK_VOL_COLS].to_numpy()
    bid_prices = df_inst[BID_PRICE_COLS].to_numpy()
    bid_vols = df_inst[BID_VOL_COLS].to_numpy()
    close_price = df_inst['close'].dropna().iloc[-1]
    
    # Use predictive_scheduler to build positions
    mid_preds = val_preds[i]
    scheduler_positions = build_schedule_from_forecasts(
        mid_pred=mid_preds,
        volume_to_fill=volume_to_fill,
        spread_pred=None,  # We don't predict spread
        liq_pred=None,     # We don't predict liquidity
        cfg=sched_cfg
    )
    
    # Simulate
    sched_vol, sched_avg_price = simulate_walk_the_book(
        scheduler_positions, ask_prices, ask_vols, bid_prices, bid_vols
    )
    
    if sched_vol > 0 and not np.isnan(sched_avg_price):
        impl_error = np.abs(sched_avg_price - close_price) / close_price * 10000
        vol_penalty = min(100.0, volume_to_fill / sched_vol)
        scheduler_bps.append(impl_error * vol_penalty)

scheduler_bps = np.array(scheduler_bps)

print(f"\n{'='*50}")
print(f"PREDICTIVE SCHEDULER IMPLEMENTATION ERROR")
print(f"{'='*50}")
print(f"Config: window={sched_cfg.window}, alpha={sched_cfg.alpha}")
print(f"Instruments evaluated: {len(scheduler_bps)}")
print(f"Mean:   {scheduler_bps.mean():.4f} bps")
print(f"Median: {np.median(scheduler_bps):.4f} bps")
print(f"Std:    {scheduler_bps.std():.4f} bps")
print(f"Min:    {scheduler_bps.min():.4f} bps")
print(f"Max:    {scheduler_bps.max():.4f} bps")
print(f"{'='*50}")

/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 55 level 5. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 48 level 3. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 48 level 4. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 48 level 5. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 57 level 3. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encountered in ASK book at second 57 level 4. Skipping level.
  warnings.warn(
/root/data/simulate_walk_the_book.py:52: UserWarning: NaN price or volume encounte


PREDICTIVE SCHEDULER IMPLEMENTATION ERROR
Config: window=14, alpha=0.1
Instruments evaluated: 136
Mean:   1.3234 bps
Median: 1.2364 bps
Std:    1.1250 bps
Min:    0.0004 bps
Max:    9.9546 bps
